## 멀티 체인 

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

In [ ]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm_first = get_llm(temperature=0.2)
llm_second = get_llm(temperature=0.7)
print(llm_first.model_name)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 세계 여행 전문가입니다.\n"
     "사용자가 말한 여행지의 가장 대표적인 관광 명소 1곳을 추천하세요.\n"
     "규칙: 명소의 한국어 이름만 출력하세요. 설명, 문장, 따옴표, 마침표는 절대 넣지 마세요."),
    ("user", "여행지: {city}"),
])

chain1 = (
    prompt1
    | llm_first
    | StrOutputParser()
    | (lambda s: s.strip().strip('"\'').strip())
)

print(repr(chain1.invoke({"city": "로마"})))

In [ ]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 친절한 여행 가이드입니다.\n"
     "사용자가 말한 관광 명소에 대해 아래 형식으로 한국어로 설명하세요.\n\n"
     "명소: (명소 이름)\n\n"
     "역사:\n(2~3문장)\n\n"
     "특징:\n(2~3문장)\n\n"
     "방문 팁:\n- (팁 1)\n- (팁 2)\n- (팁 3)"),
    ("user", "관광 명소: {place}"),
])

chain2 = prompt2 | llm_second | StrOutputParser()

print(chain2.invoke({"place": "콜로세움"}))

In [ ]:
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    RunnablePassthrough.assign(place=chain1)
    .assign(detail=chain2)
)

city = input("여행지를 입력하세요 (예: 로마, 뉴욕, 도쿄): ").strip()

result = full_chain.invoke({"city": city})

print(f"입력: {result['city']}")
print(f"1단계 결과(추천 명소): {result['place']}")
print("\n2단계 결과(상세 정보):")
print(result["detail"])